# Calibration

**Interview Focus**: What calibration means, ECE, temperature/Platt scaling, why it matters in production.

**Key Concepts**:
- A model is **calibrated** if P(correct | predicted prob = p) = p
- Reliability diagrams visualize calibration
- ECE (Expected Calibration Error) quantifies miscalibration
- Post-hoc calibration: temperature scaling, Platt scaling

**Math Notes**:
- ECE = $\sum_{b=1}^{B} \frac{n_b}{N} |\text{acc}(b) - \text{conf}(b)|$
- Temperature scaling: $\hat{p} = \text{softmax}(z / T)$, learn T on validation set
- Platt scaling: $\hat{p} = \sigma(a \cdot z + b)$, learn a, b on validation set

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. What Does Calibration Mean?

A classifier outputs probabilities. If it says P(positive) = 0.8 for 100 samples, we expect ~80 of them to actually be positive. If only 50 are positive, the model is **overconfident**.

In [ ]:
def generate_calibrated_model(n=2000, seed=42):
    """Generate predictions from a well-calibrated model."""
    rng = np.random.RandomState(seed)
    # Predicted probabilities
    probs = rng.beta(2, 2, n)  # spread across [0, 1]
    # Labels: for calibrated model, P(y=1|p) = p
    labels = (rng.rand(n) < probs).astype(int)
    return probs, labels


def generate_overconfident_model(n=2000, seed=42):
    """Generate predictions from an overconfident model."""
    rng = np.random.RandomState(seed)
    # True probabilities
    true_probs = rng.beta(2, 2, n)
    # Model pushes probabilities toward extremes (overconfident)
    probs = np.where(true_probs > 0.5,
                     0.5 + (true_probs - 0.5) ** 0.5 * 0.5 / 0.5 ** 0.5,
                     0.5 - (0.5 - true_probs) ** 0.5 * 0.5 / 0.5 ** 0.5)
    probs = np.clip(probs, 0.01, 0.99)
    labels = (rng.rand(n) < true_probs).astype(int)
    return probs, labels


def generate_underconfident_model(n=2000, seed=42):
    """Generate predictions from an underconfident model."""
    rng = np.random.RandomState(seed)
    true_probs = rng.beta(2, 2, n)
    # Model pushes probabilities toward 0.5 (underconfident)
    probs = 0.5 + (true_probs - 0.5) * 0.4
    labels = (rng.rand(n) < true_probs).astype(int)
    return probs, labels


probs_cal, labels_cal = generate_calibrated_model()
probs_over, labels_over = generate_overconfident_model()
probs_under, labels_under = generate_underconfident_model()

print(f"Generated 2000 samples for each model type.")
print(f"Calibrated:     mean prob = {probs_cal.mean():.3f}, mean label = {labels_cal.mean():.3f}")
print(f"Overconfident:  mean prob = {probs_over.mean():.3f}, mean label = {labels_over.mean():.3f}")
print(f"Underconfident: mean prob = {probs_under.mean():.3f}, mean label = {labels_under.mean():.3f}")

## 2. Reliability Diagram (Calibration Curve)

In [ ]:
def reliability_diagram_data(probs, labels, n_bins=10):
    """Compute calibration curve data: bin predicted probs, compute actual accuracy per bin."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = []
    bin_accuracies = []
    bin_confidences = []
    bin_counts = []
    
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i + 1])
        if i == n_bins - 1:  # include right edge for last bin
            mask = (probs >= bin_edges[i]) & (probs <= bin_edges[i + 1])
        
        if mask.sum() > 0:
            bin_centers.append((bin_edges[i] + bin_edges[i + 1]) / 2)
            bin_accuracies.append(labels[mask].mean())  # actual positive rate
            bin_confidences.append(probs[mask].mean())  # avg predicted prob
            bin_counts.append(mask.sum())
        else:
            bin_centers.append((bin_edges[i] + bin_edges[i + 1]) / 2)
            bin_accuracies.append(0)
            bin_confidences.append(0)
            bin_counts.append(0)
    
    return (np.array(bin_centers), np.array(bin_accuracies),
            np.array(bin_confidences), np.array(bin_counts))


def plot_reliability_diagram(probs, labels, title='Reliability Diagram', n_bins=10, ax=None):
    """Plot a reliability diagram."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    
    centers, accs, confs, counts = reliability_diagram_data(probs, labels, n_bins)
    
    # Bar chart of calibration
    width = 1.0 / n_bins * 0.8
    ax.bar(centers, accs, width=width, alpha=0.7, color='#2196F3', edgecolor='white',
           label='Model')
    
    # Perfect calibration line
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.7, label='Perfect calibration')
    
    # Gap visualization
    for c, a, conf in zip(centers, accs, confs):
        if conf > 0:
            color = '#F44336' if a < conf else '#4CAF50'
            ax.plot([c, c], [a, c], color=color, linewidth=2, alpha=0.5)
    
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives (Accuracy)')
    ax.set_title(title)
    ax.legend(loc='upper left')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    return ax


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plot_reliability_diagram(probs_cal, labels_cal, 'Well-Calibrated', ax=axes[0])
plot_reliability_diagram(probs_over, labels_over, 'Overconfident', ax=axes[1])
plot_reliability_diagram(probs_under, labels_under, 'Underconfident', ax=axes[2])

plt.tight_layout()
plt.show()

print("Overconfident: bars below diagonal (model says 0.9 but accuracy is only 0.7).")
print("Underconfident: bars above diagonal (model says 0.6 but accuracy is 0.8).")

## 3. Expected Calibration Error (ECE)

$$ECE = \sum_{b=1}^{B} \frac{n_b}{N} |\text{acc}(b) - \text{conf}(b)|$$

Weighted average of calibration gap across bins. ECE = 0 means perfectly calibrated.

In [ ]:
def expected_calibration_error(probs, labels, n_bins=10):
    """Compute Expected Calibration Error."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n_total = len(labels)
    
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i + 1])
        if i == n_bins - 1:
            mask = (probs >= bin_edges[i]) & (probs <= bin_edges[i + 1])
        
        n_bin = mask.sum()
        if n_bin > 0:
            acc_bin = labels[mask].mean()
            conf_bin = probs[mask].mean()
            ece += (n_bin / n_total) * abs(acc_bin - conf_bin)
    
    return ece


def maximum_calibration_error(probs, labels, n_bins=10):
    """MCE: maximum calibration gap across bins."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    max_gap = 0.0
    
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i + 1])
        if i == n_bins - 1:
            mask = (probs >= bin_edges[i]) & (probs <= bin_edges[i + 1])
        
        n_bin = mask.sum()
        if n_bin > 0:
            acc_bin = labels[mask].mean()
            conf_bin = probs[mask].mean()
            max_gap = max(max_gap, abs(acc_bin - conf_bin))
    
    return max_gap


print(f"{'Model':<20} {'ECE':>8} {'MCE':>8} {'Accuracy':>10}")
print("-" * 48)
for name, probs, labels in [('Well-Calibrated', probs_cal, labels_cal),
                             ('Overconfident', probs_over, labels_over),
                             ('Underconfident', probs_under, labels_under)]:
    ece = expected_calibration_error(probs, labels)
    mce = maximum_calibration_error(probs, labels)
    acc = ((probs > 0.5) == labels).mean()
    print(f"{name:<20} {ece:>8.4f} {mce:>8.4f} {acc:>10.4f}")

print("\nECE is the go-to calibration metric. Lower = better.")
print("MCE captures worst-case miscalibration (useful for safety-critical apps).")

## 4. Why Neural Nets Are Often Overconfident

Modern deep nets tend to be overconfident because:
1. Cross-entropy training pushes predictions toward 0 or 1
2. Overparameterization + training to convergence amplifies this
3. Batch normalization and weight decay can worsen calibration

In [ ]:
# Simulate: logits from a neural net before and after applying sigmoid/softmax
np.random.seed(42)
n = 1000

# True labels
y_true = np.random.binomial(1, 0.5, n)

# Neural net logits: correct class gets higher logit, but magnitudes are large (overconfident)
logits = np.where(y_true == 1,
                  np.random.normal(2.5, 1.0, n),   # positive class logits
                  np.random.normal(-2.5, 1.0, n))   # negative class logits

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

probs_raw = sigmoid(logits)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Logit distribution
ax = axes[0]
ax.hist(logits[y_true == 1], bins=30, alpha=0.6, label='Positive class', color='#2196F3', density=True)
ax.hist(logits[y_true == 0], bins=30, alpha=0.6, label='Negative class', color='#F44336', density=True)
ax.set_xlabel('Logit Value')
ax.set_ylabel('Density')
ax.set_title('Neural Net Logit Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Resulting probability distribution
ax = axes[1]
ax.hist(probs_raw, bins=30, alpha=0.7, color='#FF9800', edgecolor='white')
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Count')
ax.set_title('Probability Distribution (Overconfident)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

ece_raw = expected_calibration_error(probs_raw, y_true)
acc_raw = ((probs_raw > 0.5) == y_true).mean()
print(f"Raw neural net: Accuracy = {acc_raw:.4f}, ECE = {ece_raw:.4f}")
print("Most probabilities are near 0 or 1 — classic overconfidence.")

## 5. Temperature Scaling

The simplest post-hoc calibration method. Divide logits by temperature T before sigmoid/softmax.

$$\hat{p} = \sigma(z / T)$$

- T > 1: softens probabilities (reduces overconfidence)
- T < 1: sharpens probabilities
- T = 1: no change

Learn T on a held-out validation set by minimizing NLL (negative log-likelihood).

In [ ]:
def nll_loss(probs, labels):
    """Negative log-likelihood (binary cross-entropy)."""
    probs = np.clip(probs, 1e-10, 1 - 1e-10)
    return -np.mean(labels * np.log(probs) + (1 - labels) * np.log(1 - probs))


def find_temperature(logits, labels, lr=0.01, n_iter=1000):
    """Find optimal temperature T by minimizing NLL on validation set.
    
    Uses simple gradient descent on T.
    """
    T = 1.0  # initialize
    best_T = T
    best_nll = float('inf')
    
    nll_history = []
    T_history = []
    
    for i in range(n_iter):
        probs = sigmoid(logits / T)
        loss = nll_loss(probs, labels)
        nll_history.append(loss)
        T_history.append(T)
        
        if loss < best_nll:
            best_nll = loss
            best_T = T
        
        # Gradient of NLL w.r.t. T (numerical)
        eps = 1e-4
        probs_plus = sigmoid(logits / (T + eps))
        loss_plus = nll_loss(probs_plus, labels)
        grad = (loss_plus - loss) / eps
        
        T = T - lr * grad
        T = max(T, 0.01)  # T must be positive
    
    return best_T, nll_history, T_history


# Split into validation (for finding T) and test
n_val = 500
logits_val, labels_val = logits[:n_val], y_true[:n_val]
logits_test, labels_test = logits[n_val:], y_true[n_val:]

# Find optimal temperature
T_opt, nll_hist, T_hist = find_temperature(logits_val, labels_val)
print(f"Optimal temperature: T = {T_opt:.4f}")
print(f"T > 1 means the model was overconfident (logits too large).")

# Apply temperature scaling to test set
probs_uncalibrated = sigmoid(logits_test)
probs_temp_scaled = sigmoid(logits_test / T_opt)

ece_before = expected_calibration_error(probs_uncalibrated, labels_test)
ece_after = expected_calibration_error(probs_temp_scaled, labels_test)
acc_before = ((probs_uncalibrated > 0.5) == labels_test).mean()
acc_after = ((probs_temp_scaled > 0.5) == labels_test).mean()

print(f"\n{'':>20} {'Before':>12} {'After T-scaling':>16}")
print("-" * 50)
print(f"{'ECE':>20} {ece_before:>12.4f} {ece_after:>16.4f}")
print(f"{'Accuracy':>20} {acc_before:>12.4f} {acc_after:>16.4f}")
print(f"\nTemperature scaling reduces ECE without changing accuracy (same decision boundary).")

In [ ]:
# Visualize: before vs after temperature scaling
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plot_reliability_diagram(probs_uncalibrated, labels_test, 'Before (Overconfident)', ax=axes[0])
plot_reliability_diagram(probs_temp_scaled, labels_test, f'After T-Scaling (T={T_opt:.2f})', ax=axes[1])

# Show temperature search
ax = axes[2]
ax.plot(T_hist[:200], nll_hist[:200], 'b-', linewidth=1.5)
ax.axvline(x=T_opt, color='red', linestyle='--', alpha=0.7, label=f'T*={T_opt:.3f}')
ax.set_xlabel('Temperature T')
ax.set_ylabel('NLL Loss')
ax.set_title('Temperature Optimization')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Temperature Effect on Probabilities

In [ ]:
# Visualize how different temperatures affect the probability distribution
temperatures = [0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(1, len(temperatures), figsize=(16, 4))

for ax, T in zip(axes, temperatures):
    probs_t = sigmoid(logits_test / T)
    ece_t = expected_calibration_error(probs_t, labels_test)
    ax.hist(probs_t, bins=30, alpha=0.7, color='#2196F3', edgecolor='white')
    ax.set_xlabel('Predicted Probability')
    ax.set_title(f'T = {T} (ECE={ece_t:.3f})')
    ax.set_xlim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Count')
plt.suptitle('Effect of Temperature on Probability Distribution', y=1.02)
plt.tight_layout()
plt.show()

print("T < 1: sharpens (more confident). T > 1: softens (less confident).")
print("Optimal T makes the distribution match the true positive rate per bin.")

## 7. Platt Scaling

More flexible than temperature scaling: learn a linear transform of the logit.

$$\hat{p} = \sigma(a \cdot z + b)$$

- Two parameters (a, b) vs one (T) for temperature scaling
- Equivalent to training a logistic regression on the logits
- Originally proposed for calibrating SVMs

In [ ]:
def platt_scaling(logits_train, labels_train, logits_test, lr=0.01, n_iter=2000):
    """Platt scaling: learn a, b such that p = sigmoid(a*z + b)."""
    a = 1.0
    b = 0.0
    best_a, best_b = a, b
    best_nll = float('inf')
    
    for i in range(n_iter):
        # Forward
        scaled = a * logits_train + b
        probs = sigmoid(scaled)
        loss = nll_loss(probs, labels_train)
        
        if loss < best_nll:
            best_nll = loss
            best_a, best_b = a, b
        
        # Gradients (numerical for clarity)
        eps = 1e-4
        loss_a = nll_loss(sigmoid((a + eps) * logits_train + b), labels_train)
        loss_b = nll_loss(sigmoid(a * logits_train + (b + eps)), labels_train)
        
        grad_a = (loss_a - loss) / eps
        grad_b = (loss_b - loss) / eps
        
        a -= lr * grad_a
        b -= lr * grad_b
    
    # Apply to test
    probs_calibrated = sigmoid(best_a * logits_test + best_b)
    return probs_calibrated, best_a, best_b


probs_platt, a_opt, b_opt = platt_scaling(logits_val, labels_val, logits_test)

ece_platt = expected_calibration_error(probs_platt, labels_test)
acc_platt = ((probs_platt > 0.5) == labels_test).mean()

print(f"Platt scaling parameters: a = {a_opt:.4f}, b = {b_opt:.4f}")
print(f"\n{'Method':<20} {'ECE':>8} {'Accuracy':>10}")
print("-" * 40)
print(f"{'Uncalibrated':<20} {ece_before:>8.4f} {acc_before:>10.4f}")
print(f"{'Temperature':<20} {ece_after:>8.4f} {acc_after:>10.4f}")
print(f"{'Platt':<20} {ece_platt:>8.4f} {acc_platt:>10.4f}")

In [ ]:
# Compare all three side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plot_reliability_diagram(probs_uncalibrated, labels_test,
                         f'Uncalibrated (ECE={ece_before:.3f})', ax=axes[0])
plot_reliability_diagram(probs_temp_scaled, labels_test,
                         f'Temperature (ECE={ece_after:.3f})', ax=axes[1])
plot_reliability_diagram(probs_platt, labels_test,
                         f'Platt (ECE={ece_platt:.3f})', ax=axes[2])

plt.tight_layout()
plt.show()

## 8. Multi-Class Temperature Scaling

In [ ]:
def softmax(logits, T=1.0):
    """Softmax with temperature."""
    scaled = logits / T
    # Numerical stability: subtract max
    shifted = scaled - np.max(scaled, axis=1, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=1, keepdims=True)


def multiclass_ece(probs, labels, n_bins=10):
    """ECE for multi-class: use max predicted probability as confidence."""
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    correct = (predictions == labels).astype(float)
    return expected_calibration_error(confidences, correct, n_bins)


def find_temperature_multiclass(logits, labels, lr=0.01, n_iter=500):
    """Find optimal T for multi-class via NLL minimization."""
    T = 1.0
    best_T = T
    best_nll = float('inf')
    
    for _ in range(n_iter):
        probs = softmax(logits, T)
        # NLL: -log(probability of correct class)
        nll = -np.mean(np.log(probs[np.arange(len(labels)), labels] + 1e-10))
        
        if nll < best_nll:
            best_nll = nll
            best_T = T
        
        # Numerical gradient
        eps = 1e-4
        probs_plus = softmax(logits, T + eps)
        nll_plus = -np.mean(np.log(probs_plus[np.arange(len(labels)), labels] + 1e-10))
        grad = (nll_plus - nll) / eps
        
        T -= lr * grad
        T = max(T, 0.01)
    
    return best_T


# Simulate multi-class problem
np.random.seed(42)
n_classes = 5
n_samples = 2000

labels_mc = np.random.randint(0, n_classes, n_samples)

# Overconfident logits
logits_mc = np.random.randn(n_samples, n_classes) * 0.5
for i in range(n_samples):
    logits_mc[i, labels_mc[i]] += np.random.uniform(3, 6)  # large margin -> overconfident

# Split
n_val_mc = 500
logits_mc_val, labels_mc_val = logits_mc[:n_val_mc], labels_mc[:n_val_mc]
logits_mc_test, labels_mc_test = logits_mc[n_val_mc:], labels_mc[n_val_mc:]

T_mc = find_temperature_multiclass(logits_mc_val, labels_mc_val)

probs_mc_before = softmax(logits_mc_test, T=1.0)
probs_mc_after = softmax(logits_mc_test, T=T_mc)

ece_mc_before = multiclass_ece(probs_mc_before, labels_mc_test)
ece_mc_after = multiclass_ece(probs_mc_after, labels_mc_test)

print(f"Multi-class ({n_classes} classes):")
print(f"Optimal T = {T_mc:.4f}")
print(f"ECE before: {ece_mc_before:.4f}")
print(f"ECE after:  {ece_mc_after:.4f}")

# Reliability diagrams
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

conf_before = np.max(probs_mc_before, axis=1)
correct_before = (np.argmax(probs_mc_before, axis=1) == labels_mc_test).astype(float)
conf_after = np.max(probs_mc_after, axis=1)
correct_after = (np.argmax(probs_mc_after, axis=1) == labels_mc_test).astype(float)

plot_reliability_diagram(conf_before, correct_before,
                         f'Before (ECE={ece_mc_before:.3f})', ax=axes[0])
plot_reliability_diagram(conf_after, correct_after,
                         f'After T={T_mc:.2f} (ECE={ece_mc_after:.3f})', ax=axes[1])

plt.tight_layout()
plt.show()

## 9. When Calibration Matters in Production

In [ ]:
# Scenario: using predicted probabilities for decision thresholds
# e.g., "flag for human review if P(fraud) > threshold"

np.random.seed(42)
n = 5000

# True fraud rate: 2%
y_true_fraud = np.random.binomial(1, 0.02, n)

# Model logits
logits_fraud = np.where(y_true_fraud == 1,
                        np.random.normal(2.0, 1.0, n),
                        np.random.normal(-2.0, 1.0, n))

# Overconfident probabilities (typical neural net)
probs_fraud_raw = sigmoid(logits_fraud)

# Calibrated probabilities
T_fraud = 2.5  # typical temperature for overconfident model
probs_fraud_cal = sigmoid(logits_fraud / T_fraud)

# Business rule: flag if P(fraud) > 0.5
threshold = 0.5

flagged_raw = probs_fraud_raw > threshold
flagged_cal = probs_fraud_cal > threshold

# Among flagged cases, what fraction are actually fraud?
precision_raw = y_true_fraud[flagged_raw].mean() if flagged_raw.sum() > 0 else 0
precision_cal = y_true_fraud[flagged_cal].mean() if flagged_cal.sum() > 0 else 0

recall_raw = y_true_fraud[flagged_raw].sum() / y_true_fraud.sum() if y_true_fraud.sum() > 0 else 0
recall_cal = y_true_fraud[flagged_cal].sum() / y_true_fraud.sum() if y_true_fraud.sum() > 0 else 0

print(f"Fraud detection (threshold = {threshold}):")
print(f"\n{'':>20} {'Uncalibrated':>15} {'Calibrated':>15}")
print("-" * 52)
print(f"{'Flagged cases':>20} {flagged_raw.sum():>15} {flagged_cal.sum():>15}")
print(f"{'Precision':>20} {precision_raw:>15.4f} {precision_cal:>15.4f}")
print(f"{'Recall':>20} {recall_raw:>15.4f} {recall_cal:>15.4f}")
print(f"\nOverconfident model flags more cases (many false positives at P>0.5).")
print(f"Calibrated model: when it says P=0.5, it means ~50% chance — more reliable for thresholding.")
print(f"\nIn production, calibration matters for:")
print(f"  1. Threshold-based decisions (fraud, spam, content moderation)")
print(f"  2. Ensemble weighting (combining models)")
print(f"  3. Active learning (selecting uncertain samples)")
print(f"  4. Risk estimation (insurance, medical diagnosis)")

## Interview Questions & Answers

---

**Q: What does calibration mean?**

A model is calibrated if its predicted probabilities match observed frequencies. If it says P(spam)=0.7 for 1000 emails, about 700 should actually be spam. Formally: P(Y=1 | p(X)=p) = p for all p in [0,1].

Calibration is different from accuracy — a model can be accurate but poorly calibrated (e.g., always predicting 0.99 for positives and 0.01 for negatives, even when the true rates are 0.8 and 0.2).

---

**Q: How to calibrate a neural net?**

1. **Temperature scaling** (simplest, most common): divide logits by learned T before softmax. Just one parameter, preserves accuracy, works well in practice.
2. **Platt scaling**: sigmoid(a*z + b) — two parameters, more flexible.
3. **Isotonic regression**: non-parametric, maps probabilities to calibrated values. More flexible but needs more data.
4. **Histogram binning**: bin predictions, replace with empirical accuracy per bin.

All require a held-out calibration set (do NOT calibrate on training data).

---

**Q: Why is calibration important for production?**

1. **Threshold decisions**: if P(fraud) > 0.5 triggers review, you need P=0.5 to actually mean 50% fraud rate.
2. **Model ensembling**: combining models requires trustworthy probabilities for proper weighting.
3. **Downstream decisions**: medical diagnosis, loan approval, autonomous driving — users need to trust the confidence level.
4. **Active learning**: selecting samples where the model is most uncertain requires calibrated uncertainty.

---

**Q: Temperature scaling vs Platt scaling?**

| | Temperature | Platt |
|--|------------|-------|
| Parameters | 1 (T) | 2 (a, b) |
| Preserves accuracy? | Yes (same argmax) | Not necessarily |
| Multi-class? | Natural extension | Typically binary |
| When to use | First choice for neural nets | Binary classifiers, SVMs |

## Quick Reference

```
Calibrated: P(correct | predicted_prob = p) = p

ECE = sum_b (n_b/N) * |acc(b) - conf(b)|     (lower = better)
MCE = max_b |acc(b) - conf(b)|                (worst-case gap)

Temperature scaling:  p = softmax(z / T)      (learn T on val set)
Platt scaling:        p = sigmoid(a*z + b)     (learn a, b on val set)

T > 1 -> softens predictions (fixes overconfidence)
T < 1 -> sharpens predictions
T = 1 -> no change
```